# Depth Anything V1 Fine-Tuning

**Part 3 of 3.** Requires the outputs of `01_Pothole440_Preparation.ipynb`. Independent of `02_MiDaS31_Finetuning.ipynb` (no MiDaS checkpoint is loaded).

This notebook initializes **Depth Anything V1 (small)** from its pretrained Hugging Face checkpoint and fine-tunes it on the Pothole-440 relative scanner-intensity targets with the corrected validity masks from Part 1.

## Experiment protocol
- Model: `LiheYoung/depth-anything-small-hf`
- Data: the corrected manifest from Part 1 (`training_manifest_corrected.csv`); no new split is created.
- Target: globally normalized scanner grayscale target (not metric depth); the normalization limits come from Part 1's `global_normalization.json`.
- Validity: corrected scanner-support mask; invalid pixels are excluded from the loss.
- Input: Depth Anything V1 image processor (518 px, aspect ratio preserved).
- Target resolution: 256 × 256 (same as the MiDaS experiment).
- Initialization: pretrained Hugging Face DA V1 Small weights; no previous fine-tuned checkpoint is loaded.
- Fine-tuning: 5 epochs; best checkpoint chosen by validation loss.
- Test set: not evaluated in this notebook.

**Setup:** enable a GPU and internet (the model is downloaded from Hugging Face). If Part 1 ran in a different Kaggle session, add its output as an input dataset and set `PREP_DIR` in section 2 to that dataset's path.

The scanner-derived target is a relative grayscale proxy, not calibrated metric depth, and the estimated validity masks are heuristic. State both limitations in the report.

## 1. Environment and imports

In [1]:
import os
import gc
import json
import random
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoImageProcessor, AutoModelForDepthEstimation

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("PyTorch:", torch.__version__)


Device: cuda
GPU: Tesla T4
PyTorch: 2.11.0+cu130


## 2. Configure paths

Part 1's manifests store absolute `/kaggle/working/...` paths. `remap()` re-roots them under `PREP_DIR` when Part 1's output is attached as a Kaggle input.

In [2]:
from pathlib import Path

# Folder that contains Pothole440_Targets_Global/ and Pothole440_Targets_Corrected/ from Part 1.
# Same session as Part 1: leave as is.
# Part 1 added as a Kaggle input: e.g. Path("/kaggle/input/pothole440-prep")
PREP_DIR = Path("/kaggle/input/datasets/siddharthk7704/pothole-dataset")

WORKING_PREFIX = "/kaggle/working/"

def remap(path):
    s = str(path)
    if PREP_DIR != Path("/kaggle/working") and s.startswith(WORKING_PREFIX):
        return str(PREP_DIR / s[len(WORKING_PREFIX):])
    return s

TARGET_ROOT = PREP_DIR / "Pothole440_Targets_Global"
CORRECTED_ROOT = PREP_DIR / "Pothole440_Targets_Corrected"
MANIFEST_PATH = CORRECTED_ROOT / "training_manifest_corrected.csv"
NORM_PATH = TARGET_ROOT / "global_normalization.json"

OUTPUT_DIR = "/kaggle/working/DepthAnythingV1_Finetuned"
PREDICTION_DIR = os.path.join(OUTPUT_DIR, "validation_predictions")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PREDICTION_DIR, exist_ok=True)

TARGET_SIZE = 256
BATCH_SIZE = 2
EPOCHS = 5
NUM_WORKERS = 0
MODEL_ID = "LiheYoung/depth-anything-small-hf"

for p in [MANIFEST_PATH, NORM_PATH]:
    assert p.is_file(), f"Missing Part 1 output: {p}. Run Part 1 or fix PREP_DIR."

with open(NORM_PATH) as f:
    _norm = json.load(f)
GLOBAL_TARGET_RANGE = [_norm["p1"], _norm["p99"]]

print("Manifest:", MANIFEST_PATH)
print("Global target range:", GLOBAL_TARGET_RANGE)
print("Output:", OUTPUT_DIR)

Manifest: /kaggle/input/datasets/siddharthk7704/pothole-dataset/Pothole440_Targets_Corrected/training_manifest_corrected.csv
Global target range: [16, 173]
Output: /kaggle/working/DepthAnythingV1_Finetuned


## 3. Load the corrected manifest

In [3]:
df = pd.read_csv(MANIFEST_PATH)
print("Manifest shape:", df.shape)

REQUIRED_COLUMNS = ["split", "filename", "rgb_path", "target_path", "corrected_valid_path"]
missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
assert not missing_cols, f"Manifest is missing columns {missing_cols}. Found: {df.columns.tolist()}"

for col in ["rgb_path", "target_path", "corrected_valid_path"]:
    df[col] = df[col].map(remap)

display(df.head(3))

Manifest shape: (435, 11)


,split,filename,rgb_path,mask_path,target_path,valid_path,height,width,valid_pixels,valid_fraction,corrected_valid_path
0,train,test__2022_05_22_pothole_10_horizantal_aug1.png,/kaggle/input/datasets/siddharthk7704/pothole-...,/kaggle/working/Pothole440_Grouped/train/mask/...,/kaggle/input/datasets/siddharthk7704/pothole-...,/kaggle/working/Pothole440_Targets_Global/trai...,800,1200,113419,0.118145,/kaggle/input/datasets/siddharthk7704/pothole-...
1,train,test__2022_05_22_pothole_11_horizantal_vertica...,/kaggle/input/datasets/siddharthk7704/pothole-...,/kaggle/working/Pothole440_Grouped/train/mask/...,/kaggle/input/datasets/siddharthk7704/pothole-...,/kaggle/working/Pothole440_Targets_Global/trai...,800,1200,115216,0.120017,/kaggle/input/datasets/siddharthk7704/pothole-...
2,train,test__2022_05_22_pothole_11_horizantal_vertica...,/kaggle/input/datasets/siddharthk7704/pothole-...,/kaggle/working/Pothole440_Grouped/train/mask/...,/kaggle/input/datasets/siddharthk7704/pothole-...,/kaggle/working/Pothole440_Targets_Global/trai...,800,1200,115376,0.120183,/kaggle/input/datasets/siddharthk7704/pothole-...


## 4. Verify paths and the split

Only the train and validation partitions are used. The test partition is left untouched.

In [4]:
path_checks = {
    col: int((~df[col].map(os.path.isfile)).sum())
    for col in ["rgb_path", "target_path", "corrected_valid_path"]
}
print("Missing-path counts:", path_checks)
assert sum(path_checks.values()) == 0, "Some files are missing. Check PREP_DIR and Part 1 outputs."

print("Manifest split counts:")
display(df["split"].value_counts().rename_axis("split").to_frame("images"))

train_df = df[df["split"] == "train"].copy().reset_index(drop=True)
val_df = df[df["split"] == "val"].copy().reset_index(drop=True)

print("Train images:", len(train_df))
print("Validation images:", len(val_df))
assert len(train_df) > 0 and len(val_df) > 0
assert (df["split"] == "test").any(), "Expected a test partition in the manifest (kept unused)."

Missing-path counts: {'rgb_path': 0, 'target_path': 0, 'corrected_valid_path': 0}
Manifest split counts:


,images
split,
train,305
val,66
test,64


Train images: 305
Validation images: 66


## 5. Load the Depth Anything V1 processor and model

In [5]:
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
da1_model = AutoModelForDepthEstimation.from_pretrained(MODEL_ID).to(DEVICE)

print("Model:", MODEL_ID)
print("Parameters (M):", sum(p.numel() for p in da1_model.parameters()) / 1e6)
print("Processor size:", processor.size)


preprocessor_config.json:   0%|          | 0.00/437 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/954 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 99.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

Model: LiheYoung/depth-anything-small-hf
Parameters (M): 24.785089
Processor size: SizeDict(height=518, width=518, longest_edge=None, shortest_edge=None, max_height=None, max_width=None, min_pixels=None, max_pixels=None)


## 6. Dataset and dataloaders

The RGB image is processed by the DA V1 processor. Target and validity mask are resized to 256 × 256. The validity mask is used to exclude unsupported pixels from the loss.

In [6]:
class DA1CorrectedDataset(Dataset):
    def __init__(self, dataframe, image_processor, target_size=256):
        self.df = dataframe.reset_index(drop=True)
        self.processor = image_processor
        self.target_size = target_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["rgb_path"]).convert("RGB")

        # Part 1 stores targets as float .npy arrays (not images) and the
        # validity masks as PNGs. Loading/resizing matches the MiDaS notebook (02).
        target = np.load(row["target_path"]).astype(np.float32)
        valid = cv2.imread(row["corrected_valid_path"], cv2.IMREAD_GRAYSCALE)
        assert valid is not None, f"Unreadable validity mask: {row['corrected_valid_path']}"
        assert target.shape == valid.shape, (
            f"Target/mask shape mismatch for {row['filename']}: "
            f"{target.shape} vs {valid.shape}"
        )

        valid = (valid > 127).astype(np.float32)

        target = cv2.resize(
            target, (self.target_size, self.target_size),
            interpolation=cv2.INTER_LINEAR
        )
        valid = cv2.resize(
            valid, (self.target_size, self.target_size),
            interpolation=cv2.INTER_NEAREST
        )

        pixel_values = self.processor(
            images=image,
            return_tensors="pt"
        )["pixel_values"].squeeze(0)

        return {
            "pixel_values": pixel_values,
            "target": torch.from_numpy(target),
            "valid": torch.from_numpy(valid),
            "filename": str(row["filename"])
        }

train_dataset = DA1CorrectedDataset(train_df, processor, TARGET_SIZE)
val_dataset = DA1CorrectedDataset(val_df, processor, TARGET_SIZE)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda")
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda")
)

sample = train_dataset[0]
print("Input tensor:", tuple(sample["pixel_values"].shape))
print("Target tensor:", tuple(sample["target"].shape))
print("Validity tensor:", tuple(sample["valid"].shape))
print("Valid fraction:", float(sample["valid"].mean()))


Input tensor: (3, 518, 784)
Target tensor: (256, 256)
Validity tensor: (256, 256)
Valid fraction: 0.1184844970703125


## 7. Calibration layer

The model loaded in section 5 comes from the public pretrained DA V1 Small checkpoint; no earlier Pothole-440 fine-tuned checkpoint is used.

The affine calibration layer maps DA V1's relative output to the prepared normalized scanner-intensity target. Its initial scale and bias are estimated using **training data only**. Validation and test data are not used for this initialization.


In [7]:
class DA1Calibration(nn.Module):
    def __init__(self, scale=1.0, bias=0.0):
        super().__init__()
        self.scale = nn.Parameter(torch.tensor(float(scale)))
        self.bias = nn.Parameter(torch.tensor(float(bias)))

    def forward(self, x):
        return self.scale * x + self.bias

da1_calibration = DA1Calibration().to(DEVICE)

print("Calibration layer ready (scale=1, bias=0 until fitted in section 9).")


Calibration layer ready (scale=1, bias=0 until fitted in section 9).


## 8. Forward pass and masked loss

In [8]:
def da1_forward(pixel_values):
    outputs = da1_model(pixel_values=pixel_values)
    raw = outputs.predicted_depth

    if raw.ndim == 3:
        raw = raw.unsqueeze(1)

    raw = F.interpolate(
        raw,
        size=(TARGET_SIZE, TARGET_SIZE),
        mode="bilinear",
        align_corners=False
    ).squeeze(1)

    return da1_calibration(raw.float())

def masked_smooth_l1(prediction, target, valid):
    loss_map = F.smooth_l1_loss(prediction, target, reduction="none")
    valid = valid.float()
    denominator = valid.sum().clamp_min(1.0)
    return (loss_map * valid).sum() / denominator

# Quick shape and finite-value check.
batch = next(iter(train_loader))
with torch.no_grad():
    test_pred = da1_forward(batch["pixel_values"].to(DEVICE))
print("Prediction shape:", tuple(test_pred.shape))
print("Target shape:", tuple(batch["target"].shape))
print("Prediction finite:", bool(torch.isfinite(test_pred).all()))
print("Initial corrected-mask loss:", masked_smooth_l1(
    test_pred,
    batch["target"].to(DEVICE),
    batch["valid"].to(DEVICE)
).item())
del batch, test_pred
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Prediction shape: (2, 256, 256)
Target shape: (2, 256, 256)
Prediction finite: True
Initial corrected-mask loss: 10.550582885742188


## 9. Initialize affine calibration from training samples only

DA V1 outputs are not on the same numeric scale as the prepared target. We estimate an initial affine mapping \(y = ax+b\) from a bounded sample of valid training pixels. The fitted parameters are only an initialization; both parameters remain trainable during fine-tuning.

In [9]:
# Collect a bounded sample of valid raw predictions and targets from a fixed,
# seeded random subset of TRAINING images only (no validation/test information).
from torch.utils.data import Subset

fit_rng = np.random.default_rng(SEED)
fit_indices = fit_rng.permutation(len(train_dataset))[: min(len(train_dataset), 40 * BATCH_SIZE)]
fit_loader = DataLoader(
    Subset(train_dataset, fit_indices.tolist()),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)

da1_model.eval()
raw_samples = []
target_samples = []
max_pixels_per_batch = 20000
max_batches_for_fit = len(fit_loader)

with torch.no_grad():
    for batch_idx, batch in enumerate(tqdm(fit_loader, total=max_batches_for_fit,
                                           desc="Fit DA1 calibration from train")):
        if batch_idx >= max_batches_for_fit:
            break

        pixel_values = batch["pixel_values"].to(DEVICE, non_blocking=True)
        target = batch["target"].to(DEVICE, non_blocking=True)
        valid = batch["valid"].to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            outputs = da1_model(pixel_values=pixel_values)
            raw = outputs.predicted_depth
            if raw.ndim == 3:
                raw = raw.unsqueeze(1)
            raw = F.interpolate(
                raw, size=(TARGET_SIZE, TARGET_SIZE),
                mode="bilinear", align_corners=False
            ).squeeze(1)

        raw = raw.float()
        for j in range(raw.shape[0]):
            mask = valid[j] > 0.5
            if mask.sum() == 0:
                continue
            x = raw[j][mask].detach().cpu().numpy().astype(np.float64)
            y = target[j][mask].detach().cpu().numpy().astype(np.float64)

            if len(x) > max_pixels_per_batch:
                rng = np.random.default_rng(SEED + batch_idx * 100 + j)
                chosen = rng.choice(len(x), size=max_pixels_per_batch, replace=False)
                x, y = x[chosen], y[chosen]

            raw_samples.append(x)
            target_samples.append(y)

if not raw_samples:
    raise RuntimeError("No valid training pixels found for DA V1 calibration.")

x_fit = np.concatenate(raw_samples)
y_fit = np.concatenate(target_samples)

# Least-squares affine fit: y ~= scale * raw_prediction + bias.
A = np.column_stack([x_fit, np.ones_like(x_fit)])
scale_init, bias_init = np.linalg.lstsq(A, y_fit, rcond=None)[0]

da1_calibration.scale.data.fill_(float(scale_init))
da1_calibration.bias.data.fill_(float(bias_init))

calibrated_fit = scale_init * x_fit + bias_init
fit_mae = float(np.mean(np.abs(calibrated_fit - y_fit)))
fit_rmse = float(np.sqrt(np.mean((calibrated_fit - y_fit) ** 2)))

print("Training pixels used:", len(x_fit))
print("Initial affine scale:", float(scale_init))
print("Initial affine bias:", float(bias_init))
print("Train-sample calibration MAE:", fit_mae)
print("Train-sample calibration RMSE:", fit_rmse)

# Release calibration sample arrays before training.
del raw_samples, target_samples, x_fit, y_fit, A, calibrated_fit
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Fit DA1 calibration from train:   0%|          | 0/40 [00:00<?, ?it/s]

Training pixels used: 984530
Initial affine scale: -0.0018321263631904492
Initial affine bias: 0.7099853029407297
Train-sample calibration MAE: 0.12045013173021461
Train-sample calibration RMSE: 0.16813094996430133


## 10. Validation function

In [10]:
@torch.no_grad()
def validate_da1():
    da1_model.eval()
    da1_calibration.eval()

    losses = []
    for batch in val_loader:
        pixel_values = batch["pixel_values"].to(DEVICE, non_blocking=True)
        target = batch["target"].to(DEVICE, non_blocking=True)
        valid = batch["valid"].to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            prediction = da1_forward(pixel_values)

        loss = masked_smooth_l1(prediction.float(), target, valid)
        losses.append(float(loss.item()))

    return float(np.mean(losses))


## 11. Fine-tune with corrected validity masks

Five epochs are used as the initial training budget. The best checkpoint is selected by validation loss. If validation loss starts rising, the best checkpoint is still retained.

In [11]:
optimizer = torch.optim.AdamW(
    [
        {"params": da1_model.parameters(), "lr": 1e-5},
        {"params": da1_calibration.parameters(), "lr": 1e-3},
    ],
    weight_decay=1e-4
)

scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

best_val_loss = float("inf")
history = []

for epoch in range(1, EPOCHS + 1):
    da1_model.train()
    da1_calibration.train()

    train_losses = []
    progress = tqdm(train_loader, desc=f"Corrected DA V1 Epoch {epoch}/{EPOCHS}")

    for batch in progress:
        pixel_values = batch["pixel_values"].to(DEVICE, non_blocking=True)
        target = batch["target"].to(DEVICE, non_blocking=True)
        valid = batch["valid"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            prediction = da1_forward(pixel_values)
            loss = masked_smooth_l1(prediction.float(), target, valid)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            list(da1_model.parameters()) + list(da1_calibration.parameters()),
            max_norm=1.0
        )
        scaler.step(optimizer)
        scaler.update()

        train_losses.append(float(loss.item()))
        progress.set_postfix(loss=f"{loss.item():.5f}")

    train_loss = float(np.mean(train_losses))
    val_loss = validate_da1()

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "scale": float(da1_calibration.scale.item()),
        "bias": float(da1_calibration.bias.item())
    })

    print(
        f"Epoch {epoch}: Train={train_loss:.6f}, "
        f"Val={val_loss:.6f}, "
        f"Scale={da1_calibration.scale.item():.6f}, "
        f"Bias={da1_calibration.bias.item():.6f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            "epoch": epoch,
            "model_state_dict": da1_model.state_dict(),
            "calibration_state_dict": da1_calibration.state_dict(),
            "val_loss": val_loss,
            "train_loss": train_loss,
            "model_id": MODEL_ID,
            "target_size": TARGET_SIZE,
            "global_target_range": GLOBAL_TARGET_RANGE,
            "protocol": "corrected scanner-support validity masks"
        }, os.path.join(OUTPUT_DIR, "best_checkpoint.pth"))
        print("Saved best corrected DA V1 checkpoint.")

    pd.DataFrame(history).to_csv(
        os.path.join(OUTPUT_DIR, "training_history.csv"),
        index=False
    )

print("\nCorrected DA V1 fine-tuning complete.")
print("Best validation loss:", best_val_loss)


Corrected DA V1 Epoch 1/5:   0%|          | 0/153 [00:00<?, ?it/s]

Epoch 1: Train=0.013742, Val=0.010410, Scale=-0.006678, Bias=0.703083
Saved best corrected DA V1 checkpoint.


Corrected DA V1 Epoch 2/5:   0%|          | 0/153 [00:00<?, ?it/s]

Epoch 2: Train=0.011943, Val=0.009627, Scale=-0.009869, Bias=0.746065
Saved best corrected DA V1 checkpoint.


Corrected DA V1 Epoch 3/5:   0%|          | 0/153 [00:00<?, ?it/s]

Epoch 3: Train=0.010460, Val=0.008973, Scale=-0.016174, Bias=0.780976
Saved best corrected DA V1 checkpoint.


Corrected DA V1 Epoch 4/5:   0%|          | 0/153 [00:00<?, ?it/s]

Epoch 4: Train=0.008817, Val=0.008439, Scale=-0.018240, Bias=0.815778
Saved best corrected DA V1 checkpoint.


Corrected DA V1 Epoch 5/5:   0%|          | 0/153 [00:00<?, ?it/s]

Epoch 5: Train=0.006982, Val=0.010772, Scale=-0.020558, Bias=0.839621

Corrected DA V1 fine-tuning complete.
Best validation loss: 0.008439362151407157


## 12. Evaluate the best corrected DA V1 checkpoint

Reload the best validation checkpoint before evaluation. This cell evaluates validation only; the test partition remains untouched.

In [12]:
best_ckpt_path = os.path.join(OUTPUT_DIR, "best_checkpoint.pth")
best_ckpt = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)

da1_model.load_state_dict(best_ckpt["model_state_dict"])
da1_calibration.load_state_dict(best_ckpt["calibration_state_dict"])
da1_model.eval()
da1_calibration.eval()

print("Loaded best corrected DA V1 checkpoint.")
print("Best epoch:", best_ckpt["epoch"])
print("Best validation loss:", best_ckpt["val_loss"])

results = []
all_abs_errors = []
all_sq_errors = []
all_valid_counts = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Evaluating corrected DA V1"):
        pixel_values = batch["pixel_values"].to(DEVICE, non_blocking=True)
        target_batch = batch["target"].to(DEVICE, non_blocking=True)
        valid_batch = batch["valid"].to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            predictions = da1_forward(pixel_values)

        predictions = predictions.float()

        for i, filename in enumerate(batch["filename"]):
            pred = predictions[i].cpu().numpy()
            target = target_batch[i].cpu().numpy()
            valid = valid_batch[i].cpu().numpy() > 0.5

            if valid.sum() == 0:
                continue

            error = pred[valid] - target[valid]
            abs_error = np.abs(error)

            mae = float(abs_error.mean())
            rmse = float(np.sqrt(np.mean(error ** 2)))

            results.append({
                "filename": filename,
                "valid_pixels": int(valid.sum()),
                "MAE": mae,
                "RMSE": rmse
            })

            all_abs_errors.append(abs_error)
            all_sq_errors.append(error ** 2)
            all_valid_counts.append(int(valid.sum()))

            pred_save = np.clip(pred, 0, 1)
            pred_img = (pred_save * 255).astype(np.uint8)
            Image.fromarray(pred_img).save(
                os.path.join(PREDICTION_DIR, filename)
            )

metrics_df = pd.DataFrame(results)
metrics_df.to_csv(
    os.path.join(OUTPUT_DIR, "validation_metrics.csv"),
    index=False
)

total_valid = int(metrics_df["valid_pixels"].sum())
pixel_mae = float(sum(x.sum() for x in all_abs_errors) / total_valid)
pixel_rmse = float(np.sqrt(sum(x.sum() for x in all_sq_errors) / total_valid))

print("\nCorrected DA V1 validation results")
print("-----------------------------------")
print("Images evaluated:", len(metrics_df))
print("Valid pixels:", total_valid)
print("Pixel-weighted MAE:", pixel_mae)
print("Pixel-weighted RMSE:", pixel_rmse)
print("Image-wise mean MAE:", metrics_df["MAE"].mean())
print("Image-wise mean RMSE:", metrics_df["RMSE"].mean())
print("Median image MAE:", metrics_df["MAE"].median())
print("Median image RMSE:", metrics_df["RMSE"].median())

print("\nTen highest-error validation samples:")
display(metrics_df.sort_values("MAE", ascending=False).head(10))


Loaded best corrected DA V1 checkpoint.
Best epoch: 4
Best validation loss: 0.008439362151407157


Evaluating corrected DA V1:   0%|          | 0/33 [00:00<?, ?it/s]


Corrected DA V1 validation results
-----------------------------------
Images evaluated: 66
Valid pixels: 1197528
Pixel-weighted MAE: 0.08598604053258896
Pixel-weighted RMSE: 0.121938556432724
Image-wise mean MAE: 0.09432898326353593
Image-wise mean RMSE: 0.12318139506334608
Median image MAE: 0.08952541649341583
Median image RMSE: 0.11188692599534988

Ten highest-error validation samples:


,filename,valid_pixels,MAE,RMSE
40,train__2022_06_03_pothole_28_horizantal.png,27212,0.177977,0.206173
39,train__2022_06_03_pothole_28.png,27204,0.172101,0.199584
41,train__2022_06_03_pothole_28_vertical.png,27078,0.170149,0.195599
3,test__2022_06_03_pothole_28_horizantal_vertica...,27087,0.169456,0.196530
44,train__2022_06_03_pothole_30_horizantal_vertic...,12095,0.158213,0.183820
45,train__2022_06_03_pothole_30_vertical.png,12091,0.152663,0.180259
43,train__2022_06_03_pothole_30_horizantal.png,12092,0.148698,0.174722
42,train__2022_06_03_pothole_30.png,12088,0.145891,0.171418
60,val__2022_05_22_pothole_23_horizantal_aug53.png,8742,0.144741,0.209570
11,train__2022_05_22_pothole_23.png,8760,0.142466,0.210481


## 13. Save experiment summary

In [13]:
summary = {
    "model": "Depth Anything V1 Small",
    "model_id": MODEL_ID,
    "best_epoch": int(best_ckpt["epoch"]),
    "best_validation_loss": float(best_ckpt["val_loss"]),
    "images_evaluated": int(len(metrics_df)),
    "valid_pixels": int(total_valid),
    "pixel_weighted_MAE": float(pixel_mae),
    "pixel_weighted_RMSE": float(pixel_rmse),
    "image_mean_MAE": float(metrics_df["MAE"].mean()),
    "image_mean_RMSE": float(metrics_df["RMSE"].mean()),
    "image_median_MAE": float(metrics_df["MAE"].median()),
    "image_median_RMSE": float(metrics_df["RMSE"].median()),
    "initialization": "pretrained Hugging Face DA V1 Small; no prior Pothole-440 fine-tuned checkpoint",
    "calibration_initialization": "affine least-squares fit using valid training pixels only",
    "target_interpretation": "globally normalized scanner grayscale proxy; not metric depth",
    "validity_mask": "corrected heuristic scanner-support mask",
    "global_target_range": GLOBAL_TARGET_RANGE,
    "target_size": TARGET_SIZE,
    "epochs": EPOCHS,
    "test_evaluated": False
}

with open(os.path.join(OUTPUT_DIR, "experiment_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("Saved outputs:")
for name in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", name)


Saved outputs:
 - best_checkpoint.pth
 - experiment_summary.json
 - training_history.csv
 - validation_metrics.csv
 - validation_predictions


## 14. Notes for the final comparison

- Use the **best validation checkpoint** for this model.
- Do not evaluate or tune on the test split yet.
- Use the same corrected manifest, target files, 256 × 256 target resolution and validity-mask protocol for MiDaS (02), DA V1 (this notebook), DA V2 and YOLO26-Depth.
- Compare pixel-weighted and image-wise metrics separately.
- The normalized target is a scanner-grayscale proxy, not metric depth. A lower loss indicates better agreement with this prepared target, not necessarily better physical depth estimation.
- Record the exact checkpoint, processor, image resolution, training epochs and calibration parameters for reproducibility (saved in `experiment_summary.json` and the checkpoint).